# 👉 Урок 15 & 16. Чанкинг и реранкинг

## 🎯 Цели урока

- понять, что такое чанкинг и зачем он нужен в RAG-системах;
- изучить основные стратегии и методы чанкинга;
- **научиться выбирать правильную стратегию для разных типов данных**;
- освоить работу с HTML и OCR-документами;
- познакомиться с библиотеками для чанкинга (spaCy, LangChain, NLTK);
- познакомиться с реранкером.

---

Автор курса: Логинов Дмитрий Владимирович, преподаватель и методист МШП.

## 📖 Часть 1. Что такое чанкинг и зачем он нужен?

### Определение

**Чанкинг (Chunking)** — это процесс разбиения больших текстовых документов на небольшие фрагменты (чанки), которые потом будут использоваться для поиска и генерации ответов в RAG-системах.

### Почему это важно?

```text
Документ (1000 страниц)
    ↓
Чанкинг
    ↓
Чанк 1 | Чанк 2 | ... | Чанк N
    ↓
Каждый чанк → Эмбеддинг → Поиск
```

**Проблемы без чанкинга:**

1. **Слишком большие куски** → эмбеддинг "размывается", теряется точность
2. **Слишком маленькие куски** → теряется контекст, ответы становятся бессвязными
3. **Неправильная граница** → релевантная информация может быть разорвана

### Ключевая дилемма чанкинга

```text
Маленькие чанки                Большие чанки
    ↓                              ↓
Точный поиск                   Больше контекста
    ↓                              ↓
Но теряется контекст          Но шум в эмбеддинге
```

> 💡 **Золотое правило:** чанк должен содержать *законченную смысловую единицу*.

## 🧩 Часть 2. Виды чанкинга и когда их использовать

### 2.1. Фиксированный размер (Fixed-size Chunking)

**Принцип:** разбиваем текст на куски по N символов или токенов.

**✅ Когда использовать:**
- Однородные тексты без сложной структуры (новостные ленты, посты в соцсетях)
- Когда скорость важнее качества
- На этапе прототипирования
- Для текстов, где структура не важна (например, логи чатов)

**❌ Когда НЕ использовать:**
- Структурированные документы (книги, инструкции, законы)
- Техническая документация с заголовками
- Когда важна смысловая связность

---

### 2.2. Семантический чанкинг (Semantic Chunking)

**Принцип:** разбиваем текст по смысловым границам (заголовки, абзацы, предложения).

```text
Документ:
    Заголовок 1
        Абзац 1
        Абзац 2
    Заголовок 2
        Абзац 3
        Абзац 4

Результат чанкинга:
    Чанк 1: Заголовок 1 + Абзац 1
    Чанк 2: Заголовок 1 + Абзац 2
    Чанк 3: Заголовок 2 + Абзац 3
    Чанк 4: Заголовок 2 + Абзац 4
```

**✅ Когда использовать:**
- Книги, статьи, документация
- Тексты с четкой структурой (абзацы, заголовки)
- Когда нужно сохранить логику повествования
- Образовательные материалы

**❌ Когда НЕ использовать:**
- Хаотичные тексты без структуры
- Списки, таблицы, код
- Когда структура текста неявная

---

### 2.3. Рекурсивный чанкинг (Recursive Chunking)

**Принцип:** пытаемся разбить по более крупным единицам, а если не получается — уменьшаем уровень.

```text
Уровень 1: Разбиваем по абзацам
    Если абзац > max_size:
Уровень 2: Разбиваем по предложениям
    Если предложение > max_size:
Уровень 3: Разбиваем по словам
```

**✅ Когда использовать:**
- Смешанные тексты (есть и заголовки, и длинные абзацы)
- Универсальный вариант для большинства случаев
- Когда точно неизвестна структура текста

**❌ Когда НЕ использовать:**
- Когда точно известна структура (лучше использовать семантический)
- Для очень коротких текстов
- Когда нужно строгое сохранение иерархии

---

### 2.4. Чанкинг по структуре документа (Document Structure Chunking)

**Принцип:** сохраняем иерархию документа с метаданными.

```text
Глава 1: Введение
    Раздел 1.1: История
        Абзац 1
        Абзац 2
    Раздел 1.2: Определение
        Абзац 3

Чанк 1: {chapter: "Введение", section: "1.1", text: "Абзац 1"}
Чанк 2: {chapter: "Введение", section: "1.1", text: "Абзац 2"}
Чанк 3: {chapter: "Введение", section: "1.2", text: "Абзац 3"}
```

**✅ Когда использовать:**
- Техническая документация
- Законы и нормативные акты
- Учебники и методички
- Любые документы с четкой иерархией

**❌ Когда НЕ использовать:**
- Простые тексты без структуры
- Когда структура не важна для поиска
- Для новостных статей

---

### 2.5. Скользящее окно (Sliding Window)

**Принцип:** каждый следующий чанк начинается не с начала предыдущего, а со смещением.

```text
Чанк 1: [предложение 1] [предложение 2] [предложение 3]
Чанк 2: [предложение 2] [предложение 3] [предложение 4]
Чанк 3: [предложение 3] [предложение 4] [предложение 5]
```

**✅ Когда использовать:**
- Когда важно не потерять контекст
- Для научных статей с тесной связью между абзацами
- Когда границы между темами размыты
- Для улучшения качества поиска (каждый чанк имеет контекст)

**❌ Когда НЕ использовать:**
- Когда важна уникальность чанков (будет дублирование)
- Для очень больших документов (увеличивает размер индекса)
- Когда структура четкая и нужно избежать дублирования

---

### 2.6. Агентный чанкинг (Agentic Chunking)

**Принцип:** LLM сама решает, где сделать разрыв.

```text
Текст → LLM: "Разбей этот текст на логические части" → Чанки
```

**✅ Когда использовать:**
- Для небольших объёмов текста (до 10-50 страниц)
- Когда качество критичнее скорости
- Для сложных текстов, где трудно определить границы автоматически
- Для создания идеальных чанков для поиска

**❌ Когда НЕ использовать:**
- Для больших объёмов (дорого и медленно)
- В production-системах с высокими требованиями к скорости
- Когда бюджет ограничен

## 📊 Часть 3. Полное сравнение стратегий

| Стратегия | Скорость | Качество | Стоимость | Сложность | Лучшее применение |
|-----------|----------|----------|-----------|-----------|-------------------|
| **Фиксированный размер** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐ | Новости, посты, логи |
| **Семантический** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | Статьи, книги, блоги |
| **Рекурсивный** | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | Универсальный вариант |
| **По структуре** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | Тех. документация, законы |
| **Скользящее окно** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ | Научные статьи, исследования |
| **Агентный** | ⭐ | ⭐⭐⭐⭐⭐ | ⭐ | ⭐⭐⭐⭐⭐ | Малые объёмы, высокое качество |

## 🧠 Часть 4. Как выбрать стратегию для вашего кейса

### Алгоритм выбора

```text
Шаг 1: Определите тип данных
    ↓
Шаг 2: Оцените требования к качеству
    ↓
Шаг 3: Оцените требования к скорости
    ↓
Шаг 4: Определите бюджет
    ↓
Шаг 5: Выберите стратегию
```

### Детальные рекомендации

#### 📰 Для новостных статей и блогов
**Лучшая стратегия: Семантический + фиксированный**
- Разбивайте по абзацам
- Если абзац слишком длинный → рекурсивное разбиение
- Перекрытие 10-20% для сохранения контекста

#### 📚 Для книг и учебников
**Лучшая стратегия: По структуре документа**
- Сохраняйте заголовки как метаданные
- Группируйте по главам и разделам
- Добавляйте номер страницы

#### 📄 Для технической документации
**Лучшая стратегия: По структуре документа + метаданные**
- Каждый раздел отдельно
- Сохраняйте вложенность (глава → раздел → подраздел)
- Добавляйте ключевые слова и теги

#### 🔬 Для научных статей
**Лучшая стратегия: Скользящее окно**
- Разбивайте по предложениям
- Используйте перекрытие для сохранения контекста цитирования
- Сохраняйте ссылки и формулы

#### 🗣️ Для диалогов и чатов
**Лучшая стратегия: Агентный или семантический**
- Сохраняйте последовательность реплик
- Группируйте по темам
- Если используется агентный подход → LLM определит границы тем

### Карта выбора стратегии

```text
                     Есть ли четкая структура?
                           /        \
                         Да         Нет
                        /            \
            Важен ли размер?     Хотите качество?
            /          \         /            \
         Да             Нет   Да             Нет
        /                \    /               \
Фиксированный    Семантический  Агентный   Рекурсивный
  размер          чанкинг      чанкинг      чанкинг
```

In [1]:
# 🔧 Пример 1: Фиксированный размер (Fixed-size Chunking)

text = "Длинный текст для разбиения на чанки. Это пример фиксированного чанкинга. " * 10
chunk_size = 100  # символов
overlap = 20      # перекрытие

chunks = []
for i in range(0, len(text), chunk_size - overlap):
    chunks.append(text[i:i + chunk_size])

print(f"Всего чанков: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"Чанк {i+1} (длина {len(chunk)}): {chunk[:50]}...")

Всего чанков: 10
Чанк 1 (длина 100): Длинный текст для разбиения на чанки. Это пример ф...
Чанк 2 (длина 100): й текст для разбиения на чанки. Это пример фиксиро...
Чанк 3 (длина 100): т для разбиения на чанки. Это пример фиксированног...


In [2]:
# 🔧 Пример 2: Рекурсивный чанкинг с LangChain

!pip install langchain-text-splitters -q

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Создаём текст с разной структурой
text = """
Заголовок: Новая технология AI меняет мир

Вчера была представлена новая технология искусственного интеллекта.
Она позволяет обрабатывать текст в 10 раз быстрее.

Разработчики утверждают, что это прорыв в области NLP.
Технология уже тестируется в нескольких компаниях.

Эксперты отмечают, что такое ускорение может изменить рынок AI.
Однако есть и скептики, которые сомневаются в результатах.
"""

# Рекурсивный сплиттер
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Чанк {i+1} (длина {len(chunk)}):\n{chunk}\n")

Чанк 1 (длина 41):
Заголовок: Новая технология AI меняет мир

Чанк 2 (длина 67):
Вчера была представлена новая технология искусственного интеллекта.

Чанк 3 (длина 50):
Она позволяет обрабатывать текст в 10 раз быстрее.

Чанк 4 (длина 54):
Разработчики утверждают, что это прорыв в области NLP.

Чанк 5 (длина 50):
Технология уже тестируется в нескольких компаниях.

Чанк 6 (длина 63):
Эксперты отмечают, что такое ускорение может изменить рынок AI.

Чанк 7 (длина 58):
Однако есть и скептики, которые сомневаются в результатах.



In [3]:
# 🔧 Пример 3: Семантический чанкинг с spaCy

!pip install spacy -q
!python -m spacy download ru_core_news_md -q

import spacy

# Загружаем модель
nlp = spacy.load("ru_core_news_md")

def semantic_chunking(text, max_chunk_size=200):
    doc = nlp(text)
    chunks = []
    current_chunk = []
    current_size = 0

    for sent in doc.sents:
        sent_len = len(sent.text)
        if current_size + sent_len > max_chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_size = 0
        current_chunk.append(sent.text)
        current_size += sent_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# Пример текста
text = """Машинное обучение — это область искусственного интеллекта.
Она позволяет компьютерам учиться на данных.
Глубокое обучение — это подмножество машинного обучения.
Оно использует нейронные сети с несколькими слоями.
Эти методы широко применяются в различных областях."""

chunks = semantic_chunking(text, max_chunk_size=100)

for i, chunk in enumerate(chunks):
    print(f"Чанк {i+1} (длина {len(chunk)}):\n{chunk}\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 85.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Чанк 1 (длина 60):
Машинное обучение — это область искусственного интеллекта. 


Чанк 2 (длина 46):
Она позволяет компьютерам учиться на данных. 


Чанк 3 (длина 58):
Глубокое обучение — это подмножество машинного обучения. 


Чанк 4 (длина 53):
Оно использует нейронные сети с несколькими слоями. 


Чанк 5 (длина 51):
Эти методы широко применяются в различных областях.



In [4]:
# 🔧 Пример 4: Сравнение стратегий чанкинга

import time

def fixed_chunking(text, size=100, overlap=20):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i + size])
    return chunks

def semantic_chunking_simple(text, max_size=200):
    # Простая версия - по предложениям
    sentences = text.split('. ')
    chunks = []
    current = []
    current_size = 0

    for sent in sentences:
        if current_size + len(sent) > max_size and current:
            chunks.append('. '.join(current) + '.')
            current = []
            current_size = 0
        current.append(sent)
        current_size += len(sent)

    if current:
        chunks.append('. '.join(current) + '.')
    return chunks

# Тестируем
text = "Это очень длинный текст для тестирования. " * 100

strategies = {
    "Фиксированный": fixed_chunking,
    "Семантический": semantic_chunking_simple
}

for name, strategy in strategies.items():
    start = time.time()
    chunks = strategy(text)
    elapsed = time.time() - start

    print(f"{name}:")
    print(f"  - Чанков: {len(chunks)}")
    print(f"  - Средний размер: {sum(len(c) for c in chunks) / len(chunks):.0f} символов")
    print(f"  - Время: {elapsed:.4f} сек\n")

Фиксированный:
  - Чанков: 53
  - Средний размер: 99 символов
  - Время: 0.0000 сек

Семантический:
  - Чанков: 20
  - Средний размер: 209 символов
  - Время: 0.0001 сек



## 🛠️ Часть 5. Библиотеки для чанкинга

### 5.1. LangChain — самый популярный инструмент

```python
from langchain.text_splitter import (
    CharacterTextSplitter,              # Фиксированный размер
    RecursiveCharacterTextSplitter,     # Рекурсивный
    TokenTextSplitter,                  # По токенам
    HTMLHeaderTextSplitter,             # По HTML-заголовкам
    MarkdownTextSplitter,               # По Markdown
)

# Рекурсивный сплиттер (универсальный)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_text(text)
```

### 5.2. spaCy — для продвинутого NLP

```python
import spacy
nlp = spacy.load("ru_core_news_md")

def semantic_chunking(text, max_chunk_size=500):
    doc = nlp(text)
    chunks = []
    current_chunk = []
    current_size = 0
    
    for sent in doc.sents:
        sent_len = len(sent.text)
        if current_size + sent_len > max_chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_size = 0
        current_chunk.append(sent.text)
        current_size += sent_len
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks
```

### 5.3. Сравнение библиотек

| Библиотека | Плюсы | Минусы | Когда использовать |
|------------|-------|--------|-------------------|
| **LangChain** | Много готовых сплиттеров, интеграция | Тяжёлая | Прототипирование, быстрое решение |
| **spaCy** | Быстрый, качественный NLP | Нужно загружать модели | Продвинутый семантический чанкинг |
| **NLTK** | Лёгкая, классическая | Медленнее | Простые задачи, обучение |

In [3]:
# 🔧 Пример 5: Разные сплиттеры в LangChain

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

text = "Это пример текста для демонстрации разных сплиттеров. " * 20

# 1. CharacterTextSplitter
char_splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separator=" "
)
char_chunks = char_splitter.split_text(text)

# 2. RecursiveCharacterTextSplitter
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""]
)
recursive_chunks = recursive_splitter.split_text(text)

print(f"CharacterTextSplitter: {len(char_chunks)} чанков")
print(f"RecursiveCharacterTextSplitter: {len(recursive_chunks)} чанков")
print(f"\nПример чанка из RecursiveCharacterTextSplitter:")
print(recursive_chunks[0][:100] + "...")

CharacterTextSplitter: 14 чанков
RecursiveCharacterTextSplitter: 20 чанков

Пример чанка из RecursiveCharacterTextSplitter:
Это пример текста для демонстрации разных сплиттеров...


## 🌐 Часть 6. Работа с HTML разметкой

### 6.1. Почему HTML — особенный случай?

HTML имеет встроенную структуру, которую нужно сохранять.

### 6.2. Стратегии для HTML

#### Стратегия 1: Извлечение по заголовкам

```python
from langchain.text_splitter import HTMLHeaderTextSplitter

headers_to_split_on = [
    ("h1", "Заголовок H1"),
    ("h2", "Заголовок H2"),
    ("h3", "Заголовок H3"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on)
chunks = html_splitter.split_text(html_text)

# Результат: каждый чанк содержит свой заголовок в метаданных
```

#### Стратегия 2: Очистка + чанкинг

```python
from bs4 import BeautifulSoup

def process_html(html):
    soup = BeautifulSoup(html, 'html.parser')
    
    # Удаляем мусор
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()
    
    # Извлекаем структуру
    structure = []
    for tag in soup.find_all(['h1', 'h2', 'h3', 'h4', 'p', 'li']):
        if tag.name in ['h1', 'h2', 'h3', 'h4']:
            structure.append({
                'type': 'header',
                'level': int(tag.name[1]),
                'text': tag.get_text(strip=True)
            })
        else:
            structure.append({
                'type': 'content',
                'text': tag.get_text(strip=True)
            })
    
    # Группируем контент по заголовкам
    chunks = []
    current_header = "Главная"
    current_content = []
    
    for item in structure:
        if item['type'] == 'header':
            if current_content:
                chunks.append({
                    'header': current_header,
                    'text': ' '.join(current_content)
                })
                current_content = []
            current_header = item['text']
        else:
            current_content.append(item['text'])
    
    if current_content:
        chunks.append({
            'header': current_header,
            'text': ' '.join(current_content)
        })
    
    return chunks
```

### 6.3. Когда что использовать для HTML

| Тип страницы | Стратегия | Почему |
|--------------|-----------|--------|
| **Статья/блог** | По заголовкам | Важна структура статьи |
| **Новостная лента** | Очистка + чанкинг | Структура не важна |
| **Документация** | По заголовкам + метаданные | Важна иерархия |
| **Форум/комментарии** | Семантический | Нужны смысловые блоки |

In [4]:
# 🔧 Пример 6: Обработка HTML

!pip install beautifulsoup4 lxml -q

from bs4 import BeautifulSoup

html_text = """
<html>
    <body>
        <h1>Введение в NLP</h1>
        <p>NLP — это область искусственного интеллекта.</p>
        <h2>Основные задачи</h2>
        <p>Классификация текста, NER, машинный перевод.</p>
        <h2>Современные подходы</h2>
        <p>Трансформеры и LLM произвели революцию.</p>
    </body>
</html>
"""

def extract_html_structure(html):
    soup = BeautifulSoup(html, 'html.parser')

    chunks = []
    current_header = "Главная"
    current_text = []

    for tag in soup.find_all(['h1', 'h2', 'h3', 'h4', 'p']):
        if tag.name in ['h1', 'h2', 'h3', 'h4']:
            if current_text:
                chunks.append({
                    'header': current_header,
                    'text': ' '.join(current_text)
                })
                current_text = []
            current_header = tag.get_text(strip=True)
        else:
            current_text.append(tag.get_text(strip=True))

    if current_text:
        chunks.append({
            'header': current_header,
            'text': ' '.join(current_text)
        })

    return chunks

chunks = extract_html_structure(html_text)

for i, chunk in enumerate(chunks):
    print(f"Чанк {i+1}:")
    print(f"  Заголовок: {chunk['header']}")
    print(f"  Текст: {chunk['text']}\n")

Чанк 1:
  Заголовок: Введение в NLP
  Текст: NLP — это область искусственного интеллекта.

Чанк 2:
  Заголовок: Основные задачи
  Текст: Классификация текста, NER, машинный перевод.

Чанк 3:
  Заголовок: Современные подходы
  Текст: Трансформеры и LLM произвели революцию.



## 📄 Часть 7. Работа с OCR файлами

### 7.1. Что такое OCR?

**OCR (Optical Character Recognition)** — распознавание текста с изображений и PDF-файлов.

### 7.2. Основные библиотеки

```python
# 1. Tesseract (бесплатный)
import pytesseract
from PIL import Image
text = pytesseract.image_to_string(Image.open('doc.png'), lang='rus+eng')

# 2. EasyOCR (более современный)
import easyocr
reader = easyocr.Reader(['ru', 'en'])
result = reader.readtext('doc.png')
texts = [item[1] for item in result]
```

### 7.3. Особенности OCR-текста

**Проблемы:**
1. Ошибки распознавания
2. Потеря структуры
3. Лишние пробелы
4. Неправильные абзацы

### 7.4. Стратегии чанкинга для OCR

#### Стратегия 1: Очистка → структурирование → чанкинг

```python
import re

def process_ocr_text(text):
    # 1. Очистка
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^а-яА-Яa-zA-Z0-9.,!?\s]', '', text)
    
    # 2. Восстановление структуры
    paragraphs = re.split(r'\n\s*\n', text)
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    
    # 3. Чанкинг
    chunks = []
    current = []
    for p in paragraphs:
        if len(' '.join(current + [p])) < 500:
            current.append(p)
        else:
            chunks.append(' '.join(current))
            current = [p]
    
    if current:
        chunks.append(' '.join(current))
    
    return chunks
```

### 7.5. Когда что использовать для OCR

| Тип документа | Стратегия | Почему |
|---------------|-----------|--------|
| **Отсканированная книга** | Очистка → структурирование | Сохраняем абзацы |
| **PDF с таблицами** | Метаданные + структура | Важна позиция текста |
| **Рукописный текст** | Очистка + семантический | Много ошибок распознавания |
| **Фотография документа** | Метаданные + чанкинг | Сохраняем структуру страницы |

In [5]:
# 🔧 Пример 7: Обработка OCR-текста

import re

# Симулируем OCR-текст с ошибками
ocr_text = """
Введеиие в NLP

NLP - это облсть ИИ. Она позволяет работать с текстом.

Задачи: классификация, NER, перевод. Современные методы - трансформеры.
Они очень эффективны.

Будущее NLP связано с большими моделями.
"""

def clean_and_chunk_ocr(text, max_chunk_size=200):
    # 1. Очистка
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^а-яА-Яa-zA-Z0-9.,!?\s-]', '', text)

    # 2. Восстановление абзацев
    paragraphs = re.split(r'\n\s*\n', text)
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    # 3. Чанкинг
    chunks = []
    current = []
    for p in paragraphs:
        if len(' '.join(current + [p])) < max_chunk_size:
            current.append(p)
        else:
            if current:
                chunks.append(' '.join(current))
            current = [p]

    if current:
        chunks.append(' '.join(current))

    return chunks

chunks = clean_and_chunk_ocr(ocr_text)

for i, chunk in enumerate(chunks):
    print(f"Чанк {i+1} (длина {len(chunk)}):\n{chunk}\n")

Чанк 1 (длина 203):
Введеиие в NLP NLP - это облсть ИИ. Она позволяет работать с текстом. Задачи классификация, NER, перевод. Современные методы - трансформеры. Они очень эффективны. Будущее NLP связано с большими моделями.



## 🎯 Часть 8. Практические рекомендации по выбору

### Чек-лист выбора стратегии

```text
□ Какой у вас тип данных?
  □ Новости/блоги → Семантический
  □ Книги/учебники → По структуре
  □ Тех. документация → По структуре + метаданные
  □ Научные статьи → Скользящее окно
  □ Диалоги/чаты → Агентный или семантический

□ Какие требования к качеству?
  □ Высокое → Агентный, Скользящее окно
  □ Среднее → Семантический, По структуре
  □ Низкое → Фиксированный размер

□ Какие требования к скорости?
  □ Высокие → Фиксированный размер
  □ Средние → Семантический, Рекурсивный
  □ Низкие → Агентный

□ Какой бюджет?
  □ Ограничен → Фиксированный размер, Семантический
  □ Средний → Рекурсивный, По структуре
  □ Не ограничен → Агентный
```

### Общие рекомендации

1. **Начинайте с рекурсивного чанкинга** — он работает в 80% случаев
2. **Для структурированных документов** используйте чанкинг по структуре
3. **Для сложных текстов** добавьте перекрытие (overlap) 10-20%
4. **Оптимальный размер чанка:** 200-1000 символов или 50-200 токенов
5. **Всегда тестируйте несколько стратегий** на ваших данных
6. **Собирайте метрики** для сравнения стратегий

In [6]:
# 🔧 Пример 8: Автоматический выбор стратегии

class ChunkingStrategySelector:
    def __init__(self):
        self.strategies = {
            'news': self.semantic_chunking,
            'book': self.structure_chunking,
            'tech_doc': self.structure_with_metadata,
            'research': self.sliding_window,
            'dialogue': self.agentic_chunking,
            'default': self.recursive_chunking
        }

    def semantic_chunking(self, text, **kwargs):
        return ["Семантический чанк 1", "Семантический чанк 2"]

    def structure_chunking(self, text, **kwargs):
        return ["Структурный чанк 1", "Структурный чанк 2"]

    def structure_with_metadata(self, text, **kwargs):
        return [{"text": "Чанк с метаданными", "metadata": {"section": "1.1"}}]

    def sliding_window(self, text, **kwargs):
        return ["Скользящее окно чанк 1", "Скользящее окно чанк 2"]

    def agentic_chunking(self, text, **kwargs):
        return ["Агентный чанк 1", "Агентный чанк 2"]

    def recursive_chunking(self, text, **kwargs):
        return ["Рекурсивный чанк 1", "Рекурсивный чанк 2"]

    def select_strategy(self, document_type, requirements=None):
        base = self.strategies.get(document_type, self.strategies['default'])

        if requirements and requirements.get('high_quality'):
            print(f"📈 Добавляем перекрытие для {document_type}")

        return base

# Использование
selector = ChunkingStrategySelector()

# Примеры выбора
test_cases = [
    ('news', {'high_quality': True}),
    ('book', {}),
    ('tech_doc', {'high_speed': True}),
    ('unknown', {}),
]

for doc_type, req in test_cases:
    strategy = selector.select_strategy(doc_type, req)
    result = strategy("Пример текста")
    print(f"{doc_type} → {result}")

📈 Добавляем перекрытие для news
news → ['Семантический чанк 1', 'Семантический чанк 2']
book → ['Структурный чанк 1', 'Структурный чанк 2']
tech_doc → [{'text': 'Чанк с метаданными', 'metadata': {'section': '1.1'}}]
unknown → ['Рекурсивный чанк 1', 'Рекурсивный чанк 2']


## 📊 Часть 9. Оценка качества чанкинга

### Метрики для выбора стратегии

| Метрика | Описание | Как измерять |
|---------|----------|--------------|
| **Precision@k** | Доля релевантных чанков в топ-k | Поиск по запросам |
| **Recall@k** | Доля найденных релевантных чанков | Поиск по запросам |
| **Context Coherence** | Связность текста внутри чанка | Ручная оценка / LLM |
| **Information Density** | Плотность информации в чанке | Доля значимых слов |
| **Search Speed** | Скорость поиска по чанкам | Измерение времени |

### A/B-тестирование стратегий

```python
def test_chunking_strategies(text, queries, strategies):
    results = {}
    for name, strategy in strategies.items():
        # 1. Применяем стратегию
        chunks = strategy(text)
        
        # 2. Индексируем чанки (гипотетически)
        # embeddings = embed_chunks(chunks)
        
        # 3. Измеряем качество поиска
        precision, recall = evaluate_search(queries, chunks)
        
        # 4. Измеряем скорость
        speed = measure_speed(strategy, text)
        
        results[name] = {
            'precision': precision,
            'recall': recall,
            'speed': speed,
            'num_chunks': len(chunks),
            'avg_size': sum(len(c) for c in chunks) / len(chunks)
        }
    
    return results
```

In [7]:
# 🔧 Пример 9: Оценка качества чанкинга

def evaluate_chunks(chunks, expected_content):
    """
    Простая оценка качества чанкинга
    """
    # 1. Полнота (сохранение контента)
    total_length = sum(len(c) if isinstance(c, str) else len(c.get('text', '')) for c in chunks)

    # 2. Средний размер
    sizes = [len(c) if isinstance(c, str) else len(c.get('text', '')) for c in chunks]
    avg_size = sum(sizes) / len(sizes) if sizes else 0

    # 3. Количество чанков
    num_chunks = len(chunks)

    # 4. Сохранение структуры (есть ли метаданные)
    has_metadata = any(isinstance(c, dict) and 'metadata' in c for c in chunks)

    return {
        'total_chunks': num_chunks,
        'avg_chunk_size': avg_size,
        'total_size': total_length,
        'has_metadata': has_metadata
    }

# Тестируем
test_chunks = [
    "Первый чанк с текстом.",
    "Второй чанк с текстом.",
    {
        'text': 'Третий чанк с метаданными',
        'metadata': {'section': '1.1'}
    }
]

metrics = evaluate_chunks(test_chunks, "Тестовый текст")

print("Оценка качества чанкинга:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

Оценка качества чанкинга:
  total_chunks: 3
  avg_chunk_size: 23.0
  total_size: 69
  has_metadata: True



# 🎯 Часть 10. Реранкинг (Reranking) — улучшение результатов поиска

## 📖 Что такое реранкинг?

**Реранкинг** (от англ. *rerank* — переранжировать) — это этап постобработки результатов поиска, на котором мы пересортировываем найденные документы с использованием более точных, но более дорогих моделей.

### Простая аналогия

Представьте, что вы ищете книгу в огромной библиотеке:

```text
1️⃣ Быстрый поиск (FAISS)
   └── Библиотекарь быстро пробегает по залам и приносит 100 книг,
       которые могут подойти по теме (быстро, но неточно)

2️⃣ Реранкинг
   └── Вы садитесь и внимательно просматриваете каждую из 100 книг,
       читаете аннотации и выбираете 5 самых лучших (медленно, но точно)
```

## 🤔 Зачем нужен реранкинг?

### Проблема №1: Скорость vs Точность

```text
Хотим быстро → используем приближённые методы → теряем точность
Хотим точно → используем точные методы → работаем медленно
```

**Реранкинг даёт лучшее из двух миров:**

```text
Быстрый, но неточный поиск → Медленный, но точный реранкинг
    ↓                           ↓
Находит 100 кандидатов      Выбирает 5 лучших из 100
```

### Проблема №2: Ограничения эмбеддингов

Эмбеддинги сжимают смысл текста в один вектор (например, 384 числа).

```text
📝 Длинный текст → 🔢 384 числа → ❌ Часть смысла теряется
```

При реранкинге мы используем модели, которые видят весь текст целиком, поэтому они точнее.

## 🧠 Два типа моделей для поиска

Чтобы понять реранкинг, нужно разобраться в двух подходах:

### 1. Bi-Encoder (два кодировщика)

```text
📌 Как работает:

Вопрос: "Как работает RAG?"
    ↓
Модель A → Вектор вопроса [0.1, 0.3, 0.8, ...]

Документ: "RAG — это..."
    ↓
Модель B → Вектор документа [0.2, 0.1, 0.9, ...]

Сходство = косинус между векторами → число (скор)
```

**Особенности:**
- ✅ Быстрый (можно заранее посчитать все векторы документов)
- ✅ Масштабируется на миллионы документов
- ❌ Теряет часть смысла при сжатии
- ❌ Не видит взаимодействия слов между вопросом и документом

**Используется для:** первого этапа поиска (FAISS)

### 2. Cross-Encoder (общий кодировщик)

```text
📌 Как работает:

Вопрос + Документ вместе:
"Как работает RAG? [SEP] RAG — это..."
    ↓
Одна модель → Анализирует всё вместе → Число (релевантность)
    ↓
0.95 (очень релевантно)
```

**Особенности:**
- ❌ Медленный (нельзя заранее ничего посчитать)
- ❌ Не масштабируется на миллионы документов
- ✅ Видит все взаимосвязи между словами
- ✅ Максимальная точность

**Используется для:** реранкинга (второй этап)

## ⚙️ Как работает реранкинг в RAG-пайплайне?

```text
1. Пользовательский запрос
    ↓
2. Генерация эмбеддинга запроса (Bi-Encoder)
    ↓
3. Быстрый ANN-поиск в FAISS (топ-100 кандидатов)
    ↓
4. РЕРАНКИНГ через Cross-Encoder (топ-5)
    ↓
5. Передача в LLM для генерации ответа
    ↓
6. Ответ пользователю
```

## 🔧 Практический пример реранкинга

```python
from sentence_transformers import CrossEncoder
import numpy as np

# 1. Загружаем модель для реранкинга
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Наш запрос
query = "Что такое машинное обучение?"

# 3. Документы-кандидаты (как будто их нашёл FAISS)
candidates = [
    "Машинное обучение — это раздел искусственного интеллекта",
    "Сегодня на улице солнечная погода",
    "ML позволяет компьютерам учиться на данных без программирования",
    "Глубокое обучение — это подраздел машинного обучения",
    "Рецепт борща со свёклой и капустой",
]

# 4. Реранкинг: оцениваем каждый документ
pairs = [(query, doc) for doc in candidates]
scores = model.predict(pairs)

# 5. Выводим результаты
print("Результаты реранкинга:")
for i, (doc, score) in enumerate(sorted(zip(candidates, scores), key=lambda x: -x[1])):
    print(f"{i+1}. (скор: {score:.3f}) {doc[:60]}...")
```

## 📊 Популярные модели для реранкинга

### Для английского языка

| Модель | Размер | Скорость | Точность |
|--------|--------|----------|----------|
| `ms-marco-TinyBERT-L-2-v2` | 14 MB | ⚡⚡⚡ | ⭐⭐ |
| `ms-marco-MiniLM-L-6-v2` | 90 MB | ⚡⚡ | ⭐⭐⭐ |
| `ms-marco-MiniLM-L-12-v2` | 180 MB | ⚡ | ⭐⭐⭐⭐ |
| `ms-marco-electra-base` | 420 MB | 🐢 | ⭐⭐⭐⭐⭐ |

### Для русского языка

| Модель | Размер | Особенность |
|--------|--------|-------------|
| `BAAI/bge-reranker-v2-m3` | ~1 GB | Многоязычная, лучшая для русского |
| `intfloat/multilingual-e5-small` | ~300 MB | Хорошая для русского |

## ✅ Рекомендации по использованию

### Когда использовать реранкинг?

| Сценарий | Решение |
|----------|---------|
| Мало данных (< 10k документов) | ❌ Не нужен, используйте точный поиск |
| Средние данные (10k-100k) | ✅ Лёгкий реранкинг (TinyBERT) |
| Большие данные (> 100k) | ✅ Обязательно (MiniLM) |
| Требуется максимальная точность | ✅ Тяжёлая модель (Electra) |
| Ограничение по времени (< 100 мс) | ⚠️ Только быстрый реранкинг |

### Настройка количества кандидатов

```text
Кандидатов (из FAISS) → Результат (после реранкинга)

10 кандидатов → топ-5  → реранкинг почти бесполезен
50 кандидатов → топ-5  → хорошо
100 кандидатов → топ-5 → оптимально
200 кандидатов → топ-5 → лучше, но медленнее
500 кандидатов → топ-5 → медленно, улучшение незначительное
```

### Типичные ошибки

```text
❌ Реранкинг всех документов в базе
   → Сначала FAISS (100 кандидатов), потом реранкинг

❌ Слишком мало кандидатов (5-10)
   → Минимум 50-100 кандидатов

❌ Использовать Bi-Encoder для реранкинга
   → Для реранкинга нужен Cross-Encoder

❌ Не учитывать время реранкинга
   → Всегда тестируйте latency
```

## 📈 Влияние на метрики

Экспериментальные данные (MS MARCO):

```text
┌─────────────────────┬──────────┬──────────┬──────────┐
│ Метод               │ MRR@10   │ Recall@100│ Время   │
├─────────────────────┼──────────┼──────────┼──────────┤
│ Только FAISS        │ 0.187    │ 0.853    │ 5 мс    │
│ FAISS + ReRank      │ 0.345    │ 0.857    │ 50 мс   │
│ FAISS + ReRank (big)│ 0.386    │ 0.859    │ 200 мс  │
└─────────────────────┴──────────┴──────────┴──────────┘

Вывод: Реранкинг улучшает MRR в 2 раза!
```

## 📝 Резюме

### Что мы узнали о реранкинге?

1. **Реранкинг** — это второй этап поиска, который уточняет результаты
2. **Bi-Encoder** — быстрый, но неточный (используется в FAISS)
3. **Cross-Encoder** — медленный, но точный (используется для реранкинга)
4. Реранкинг улучшает **Recall@k** на 10-50%
5. Оптимальное количество кандидатов: 50-200

### Когда использовать реранкинг?

```text
✅ Всегда, если у вас > 10k документов
✅ Всегда, если важна точность ответов
✅ Всегда, если вы используете RAG
⚠️ Только если у вас есть запас по времени (дополнительные 10-50 мс)
```

In [8]:
from sentence_transformers import CrossEncoder
import numpy as np

# 1. Загружаем модель для реранкинга
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Наш запрос
query = "Что такое машинное обучение?"

# 3. Документы-кандидаты (как будто их нашёл FAISS)
candidates = [
    "Машинное обучение — это раздел искусственного интеллекта",
    "Сегодня на улице солнечная погода",
    "ML позволяет компьютерам учиться на данных без программирования",
    "Глубокое обучение — это подраздел машинного обучения",
    "Рецепт борща со свёклой и капустой",
]

# 4. Реранкинг: оцениваем каждый документ
pairs = [(query, doc) for doc in candidates]
scores = model.predict(pairs)

# 5. Выводим результаты
print("Результаты реранкинга:")
for i, (doc, score) in enumerate(sorted(zip(candidates, scores), key=lambda x: -x[1])):
    print(f"{i+1}. (скор: {score:.3f}) {doc[:60]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Результаты реранкинга:
1. (скор: 8.067) Машинное обучение — это раздел искусственного интеллекта...
2. (скор: 7.568) Глубокое обучение — это подраздел машинного обучения...
3. (скор: 6.257) Рецепт борща со свёклой и капустой...
4. (скор: 6.052) ML позволяет компьютерам учиться на данных без программирова...
5. (скор: 6.022) Сегодня на улице солнечная погода...


## 🎯 Заключение

### Ключевые выводы

1. **Нет универсальной стратегии** — выбор зависит от данных и требований
2. **Начинайте с рекурсивного чанкинга** — это безопасный старт
3. **Для структурированных документов** используйте структуру
4. **Всегда добавляйте метаданные** — они улучшают поиск
5. **Тестируйте разные стратегии** и измеряйте качество
6. **Реранкинг улучшает точность** поиска в 2 раза без потери скорости

### Карта принятия решений

```text
                      ┌─────────────────────┐
                      │    Определите тип    │
                      │      документа       │
                      └─────────────────────┘
                               │
              ┌────────────────┼────────────────┐
              │                │                │
         ┌────┴────┐      ┌────┴────┐      ┌────┴────┐
         │ Новости │      │  Книги  │      │ Тех. док│
         └─────────┘      └─────────┘      └─────────┘
              │                │                │
         ┌────┴────┐      ┌────┴────┐      ┌────┴────┐
         │Семанти- │      │По струк-│      │Структура│
         │ческий   │      │туре     │      │+ мета-  │
         └─────────┘      └─────────┘      │данные   │
                                            └─────────┘
```

### Следующие шаги

1. Определите тип ваших данных
2. Выберите 2-3 стратегии чанкинга для тестирования
3. Реализуйте пайплайн чанкинга
4. Оцените качество с помощью метрик
5. Добавьте реранкинг для улучшения точности
6. Выберите лучшую стратегию для продакшена

---
